# Auditoría y Limpieza del Dataset de Rendimiento de Cultivos

Este notebook es el punto de partida de un proyecto más grande: entender cómo el clima histórico de Argentina (lluvias, temperaturas) impactó el rendimiento de 8 cultivos clave (avena, centeno, girasol, maní, maíz, soja, sorgo, trigo) entre 1923 y 2025. Antes de cruzar nada con el clima, necesitamos un dataset de cultivos en el que podamos confiar: sin errores de codificación, sin nombres de provincia o departamento inconsistentes, y con los valores nulos y atípicos ya entendidos (no necesariamente eliminados). Eso es lo que hacemos acá.

## 1. Librerías, Carga y Consolidación de Datos

### 1.1 Librerías

In [ ]:
import sys, os
from pathlib import Path

RUTA_PROYECTO = Path().resolve()
if RUTA_PROYECTO.name == "notebook":  # los notebooks se ejecutan desde la carpeta notebook/
    RUTA_PROYECTO = RUTA_PROYECTO.parent
sys.path.append(str(RUTA_PROYECTO))

RUTA_RAW = RUTA_PROYECTO / "data" / "raw"
RUTA_INTERIM = RUTA_PROYECTO / "data" / "interim"

import unicodedata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
from src.data_pipeline import cargar_y_consolidar_cultivos

### 1.2 Carga, homogeneización de codificación y consolidación

**Se fija `encoding="latin-1"` de forma explícita en vez de autodetectarla.** En un primer intento se usó `charset_normalizer` para detectar la codificación de cada archivo automáticamente, pero devolvió codificaciones asiáticas (`gb18030`, `cp949`) para archivos que en realidad están en español — eso corrompió el texto (provincias y departamentos quedaron ilegibles). Se verificó manualmente contra los datos reales que `latin-1` decodifica correctamente los caracteres acentuados ("Tucumán", "Córdoba", "Neuquén"), así que se fija ese valor.

La lectura, corrección de codificación y concatenación de las 8 series de cultivo se hacen en una sola función (`cargar_y_consolidar_cultivos`, en `src/data_pipeline.py`), que no modifica ni borra nada en `data/raw` — la fuente cruda queda siempre intacta — y guarda el resultado consolidado en `data/interim`.

Nótese que `soja-1ra-serie-2000-2019` y `soja-2da-serie-2000-2019` no están en `data/raw`: se decidió no incluirlas porque, tras una primera mirada, no aportan valores distintos a la serie ya incorporada `soja-serie-1941-2024` (1941-2024) — simplemente separan la misma ventana 2000-2019 en siembra temprana y tardía. Se dejaron fuera de la carpeta en vez de filtrarlas por nombre en el código: un filtro por nombre de archivo es un punto de fallo silencioso (deja de funcionar si el archivo se renombra) y no hace falta si el archivo simplemente no está.

In [ ]:
df_cultivos = cargar_y_consolidar_cultivos(
    ruta_raw=RUTA_RAW,
    ruta_interim=RUTA_INTERIM,
    nombre_final="cultivos_consolidado.csv",
)

Con los datos de los 8 cultivos ya consolidados en un solo archivo y en una única codificación, el siguiente paso es auditarlos: mirar con lupa qué hay ahí antes de cambiar o corregir nada.

## 2. Auditoría Exploratoria de Datos

### 2.1 Vista general

In [ ]:
df_cultivos.head()

In [ ]:
df_cultivos.tail(10)

### 2.2 Dimensiones y tipos de datos

In [ ]:
df_cultivos.shape

- **Dimensión del dataset concatenado: 138929 filas y 11 columnas**

In [ ]:
df_cultivos.info()

### 2.3 Valores nulos

In [ ]:
df_cultivos.isnull().sum()

- **Los valores nulos se concentran en `superficie_cosechada_ha`, `produccion_tm` y `rendimiento_kgxha`**

In [ ]:
top_nulos = df_cultivos.isnull().sum(axis=1).groupby(df_cultivos['anio']).sum().sort_values(ascending=False)
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(top_nulos)

- **Observamos que la mayor parte de los valores nulos se distribuyen en campañas anteriores a 1968. 1965 se encuentra en el top con 753**

### 2.4 Valores duplicados

In [ ]:
df_cultivos.duplicated().sum()

- **No se observan valores duplicados**

### 2.5 Estadisticos descriptivos

In [ ]:
df_cultivos.describe(include='all')

* **Se observa una incongruencia en la variable superficie_sembrada_ha, debido a que se presume que nunca su valor mínimo puede ser "0" por lo que vamos a investigar si se trata de placeholders y, de ser así, ver qué tratamiento se le da**
* **Se observan valores mínimos de "0" en las variables superficie_cosechada_ha, produccion_tm y rendimiento_kgxha, es importante destacar esto porque podría deberse a pérdidas completas de cultivos debido a condiciones climáticas extremas, información que pretendemos validar durante nuestro análisis exploratorio**
* **Valores máximos muy por encima del resto, puede deberse a errores de carga o años espectacularmente buenos y vamos a validarlo durante nuestro análisis**

In [ ]:
for col in ['superficie_cosechada_ha', 'produccion_tm', 'rendimiento_kgxha']:
    ceros = df_cultivos[col] == 0
    ceros_con_siembra = ceros & (df_cultivos['superficie_sembrada_ha'] > 0)
    print(f"{col}: {ceros.sum()} ceros totales, {ceros_con_siembra.sum()} con superficie_sembrada_ha > 0")

### 2.6 Valores atípicos

In [ ]:
Q1 = df_cultivos["rendimiento_kgxha"].quantile(0.25)
Q3 = df_cultivos["rendimiento_kgxha"].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers = df_cultivos[(df_cultivos["rendimiento_kgxha"] < limite_inferior) | (df_cultivos["rendimiento_kgxha"] > limite_superior)]

- **Se calcularon los outliers de `rendimiento_kgxha` mediante el rango intercuartílico (IQR = Q3 - Q1), considerando atípico todo valor fuera de [Q1 - 1.5·IQR, Q3 + 1.5·IQR]. No se descartan automáticamente: podrían corresponder a campañas reales excepcionales. Se visualizan por cultivo para decidir su tratamiento en la etapa de limpieza.**

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
df_cultivos.boxplot(column='rendimiento_kgxha', by='cultivo', ax=ax,
                     patch_artist=True, boxprops=dict(facecolor='#4C72B0', alpha=0.6))
ax.set_title('Distribución de rendimiento (kg/ha) por cultivo')
plt.suptitle('')
ax.set_xlabel('Cultivo')
ax.set_ylabel('Rendimiento (kg/ha)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

* **Gráfico sólo para demostrar la distribución de los datos de cada categoría, observando valores que posiblemente sean inválidos por error de carga o derivados de años espectacularmente buenos**

### 2.7 Rangos inválidos

In [ ]:
columnas_numericas = ['superficie_sembrada_ha', 'superficie_cosechada_ha', 'produccion_tm', 'rendimiento_kgxha']

for col in columnas_numericas:
    no_numericos = pd.to_numeric(df_cultivos[col], errors='coerce').isna() & df_cultivos[col].notna()
    print(col, '->', no_numericos.sum(), 'valores no numéricos')

In [ ]:
negativos = df_cultivos[(df_cultivos[columnas_numericas] < 0).any(axis=1)]

# Superficie cosechada mayor a la sembrada (no puede cosechar más de lo que sembró)
cosecha_mayor_siembra = df_cultivos[df_cultivos['superficie_cosechada_ha'] > df_cultivos['superficie_sembrada_ha']]

# Año fuera del rango esperado del dataset
anio_invalido = df_cultivos[(df_cultivos['anio'] < 1923) | (df_cultivos['anio'] > 2026)]

print('Negativos:', len(negativos))
print('Cosecha > Siembra:', len(cosecha_mayor_siembra))
print('Año inválido:', len(anio_invalido))

- **Se verificaron condiciones lógicamente imposibles, independientes de la estadística: valores negativos, año fuera de rango (1923-2025), texto no numérico oculto en columnas numéricas, y superficie cosechada mayor a la sembrada. El análisis resultó en 3 filas que posiblemente puedan tener los valores invertidos, lo vamos a verificar a continuación.**

In [ ]:
print(cosecha_mayor_siembra.to_string())

* **Corroboramos que posiblemente los datos fueron cargados erróneamente en las columnas de superficie_sembrada_ha y superficie_cosechada_ha, por lo que vamos a optar por invertirlos en lugar de eliminarlos de nuestro dataset para continuar con nuestro análisis**
* **Este análisis detectó accidentalmente que hay texto que escapó al fix de codificación que hicimos al inicio (Santo TomÃ©) debido a que son registros puntuales y aislados que estaban cargados así en sus datasets originales, a continuación vamos a tratarlo**

### 2.8 Caracteres residuales de codificación

El código anterior evidenció un caso puntual de mojibake ("Santo TomÃ©" en vez de "Santo Tomé") en la fila de soja/Corrientes/1967, que no se corrigió con el fix de codificación general. Probablemente esa fila específica ya venía en UTF-8 en el archivo fuente, mientras el resto del archivo estaba en latin-1. Se buscan más casos del mismo patrón en todas las columnas de texto antes de decidir cómo corregirlos.

In [ ]:
columnas_texto = df_cultivos.select_dtypes(include='object').columns

for col in columnas_texto:
    mojibake = df_cultivos[col].astype(str).str.contains('Ã', na=False)
    print(col, '->', mojibake.sum(), 'valores con posible mojibake')
    if mojibake.sum() > 0:
        print(df_cultivos.loc[mojibake, col].unique())

## Resumen de hallazgos pendientes para la sección de Limpieza

La auditoría nos deja una lista corta pero concreta de asuntos a resolver antes de poder confiar en estos datos:

- Invertir `superficie_sembrada_ha` / `superficie_cosechada_ha` en las 3 filas detectadas (avena 1938, maíz 1963, soja 1967)
- Corregir los casos de mojibake detectados en 2.8 (texto UTF-8 mal leído como latin-1)
- Mantener los ceros en `superficie_cosechada_ha`, `produccion_tm` y `rendimiento_kgxha` como valores válidos: >99,9% coincide con `superficie_sembrada_ha > 0`, consistente con pérdida total de cultivo
- Investigar puntualmente los ~5 casos por columna donde el cero no coincide con siembra > 0
- Revisar `superficie_sembrada_ha == 0` como posible placeholder

## 3. Limpieza y Corrección de Datos

### 3.1 Corrección de superficie sembrada/cosechada invertidas

Se detectaron en la auditoría 3 registros con `superficie_cosechada_ha` mayor a `superficie_sembrada_ha`, condición lógicamente imposible. Al inspeccionarlos, los valores resultan coherentes si se intercambian entre sí, lo que sugiere un error de carga en el orden de las columnas en la fuente original. Se corrige el orden en estos 3 registros puntuales y se verifica el cambio posteriormente.

In [ ]:
mask = df_cultivos['superficie_cosechada_ha'] > df_cultivos['superficie_sembrada_ha']

df_cultivos.loc[mask, ['superficie_sembrada_ha', 'superficie_cosechada_ha']] = \
    df_cultivos.loc[mask, ['superficie_cosechada_ha', 'superficie_sembrada_ha']].values

# Verificación: confirmar que la condición inválida ya no está presente
filas_invalidas = (df_cultivos['superficie_cosechada_ha'] > df_cultivos['superficie_sembrada_ha']).sum()
print(f'Filas con cosechada > sembrada tras la corrección: {filas_invalidas}')


### 3.2 Corrección de mojibake (codificación mal interpretada)

Durante la auditoría se detectó texto corrupto en las columnas categóricas (`cultivo`, `campania`, `provincia`, `departamento`), producto de una autodetección de codificación incorrecta al leer los archivos originales (ver `data_pipeline.py`). El patrón de corrupción es consistente con texto en `latin-1` que fue interpretado como `utf-8`.

Definimos una única función que intenta revertir ese patrón (round-trip `encode('latin-1').decode('utf-8')`). Esa misma función se usa en dos momentos: primero en modo **diagnóstico** (para medir el alcance del problema, sin modificar nada), y luego en modo **corrección** (aplicada sobre el DataFrame).

In [ ]:
def reparar_mojibake(x):
    if not isinstance(x, str):
        return x
    try:
        return x.encode('latin-1').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        return x

def diagnosticar_cambio(original, propuesto):
    """
    Compara una serie original contra una serie de valores propuestos
    (ya alineadas por índice) y devuelve cuántos valores cambiarían y
    los pares (original, propuesto) involucrados, sin modificar nada.
    Sirve para medir el alcance de cualquier transformación antes de
    aplicarla, sin importar si el valor propuesto vino de un .apply(),
    un .map() u otra fuente.
    """
    cambios = (original != propuesto) & original.notna() & propuesto.notna()
    pares = list(zip(original[cambios].unique(), propuesto[cambios].unique()))
    return cambios.sum(), pares

In [ ]:
columnas_texto = ['cultivo', 'campania', 'provincia', 'departamento']

for col in columnas_texto:
    n_cambios, pares = diagnosticar_cambio(df_cultivos[col], df_cultivos[col].apply(reparar_mojibake))
    print(f"{col}: {n_cambios} valores que cambiarían")
    if n_cambios > 0:
        print(pares)

**Resultado del diagnóstico:** [provincia: 11090 y departamento: 19202]. Se confirma mojibake en las columnas provincia y departamento, por lo que se procede a aplicar la corrección correspondiente.

In [ ]:
for col in columnas_texto:
    df_cultivos[col] = df_cultivos[col].apply(reparar_mojibake)

In [ ]:
for col in columnas_texto:
    n_cambios, _ = diagnosticar_cambio(df_cultivos[col], df_cultivos[col].apply(reparar_mojibake))
    print(f"{col}: {n_cambios} valores que cambiarían")

**Verificación:** 0 valores que cambiarían en las cuatro columnas. El mojibake fue corregido en su totalidad.

### 3.3 Estandarización de texto en columnas categóricas

Se revisan las columnas categóricas (`cultivo`, `campania`, `provincia`, `departamento`) buscando espacios en blanco sobrantes y variantes de mayúsculas/minúsculas que representen la misma categoría como si fueran distintas.

In [ ]:
columnas_categoricas = ['cultivo', 'campania', 'provincia', 'departamento']

for col in columnas_categoricas:
    unicos = df_cultivos[col].nunique()
    unicos_strip = df_cultivos[col].str.strip().nunique()
    unicos_strip_lower = df_cultivos[col].str.strip().str.lower().nunique()
    print(f"{col}: únicos={unicos} | con strip={unicos_strip} | con strip+lower={unicos_strip_lower}, {(' -> INCONSISTENCIA DETECTADA') if unicos != unicos_strip or unicos != unicos_strip_lower else ' -> No hay inconsistencias'}")

- **Se observa inconsistencia puramente de mayúsculas/minúsculas. Procedemos a detectar su procedencia**

In [ ]:
dep_lower = df_cultivos['departamento'].str.strip().str.lower()
duplicados = df_cultivos.groupby(dep_lower)['departamento'].unique()
duplicados = duplicados[duplicados.apply(len) > 1]
print(duplicados)

In [ ]:
canonico = {
    '9 de julio': '9 de Julio',
    'el dorado': 'El Dorado',
}

mask = df_cultivos['departamento'].str.strip().str.lower().isin(canonico.keys())
df_cultivos.loc[mask, 'departamento'] = (
    df_cultivos.loc[mask, 'departamento'].str.strip().str.lower().map(canonico)
)

# Verificación: debería volver a dar 455 (ya no 457)
print("Departamentos =", df_cultivos['departamento'].nunique(), "-> Cambios aplicados correctamente, ya no hay inconsistencias en la columna 'departamento'" if df_cultivos['departamento'].nunique() == 455 else "Aún hay inconsistencias en la columna 'departamento'")

In [ ]:
def quitar_acentos(texto):
    if not isinstance(texto, str):
        return texto
    return ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')

for col in columnas_categoricas:
    base = df_cultivos[col].str.strip().str.lower()
    sin_acentos = base.apply(quitar_acentos)
    print(f"{col}: únicos (strip+lower)={base.nunique()} | sin acentos={sin_acentos.nunique()} {(' -> INCONSISTENCIA DETECTADA' if base.nunique() != sin_acentos.nunique() else ' -> No hay inconsistencias')}")

In [ ]:
dep_sin_acentos = df_cultivos['departamento'].str.strip().str.lower().apply(quitar_acentos)
duplicados_acentos = df_cultivos.groupby(dep_sin_acentos)['departamento'].unique()
duplicados_acentos = duplicados_acentos[duplicados_acentos.apply(len) > 1]
print(duplicados_acentos)

In [ ]:
mask = df_cultivos['departamento'].str.strip().str.lower().apply(quitar_acentos) == 'matara'
df_cultivos.loc[mask, 'departamento'] = 'Matará'

print("Departamentos =", df_cultivos['departamento'].nunique(), "-> Cambios aplicados correctamente, ya no hay inconsistencias en la columna 'departamento'" if df_cultivos['departamento'].nunique() == 454 else "Aún hay inconsistencias en la columna 'departamento'")

In [ ]:
print(sorted(df_cultivos['provincia'].dropna().unique()))

In [ ]:
print("--- Los Andes ---")
print(df_cultivos[df_cultivos['provincia'] == 'Los Andes'][['anio', 'departamento', 'provincia_id']].drop_duplicates())

print("\n--- otras provincias ---")
print(df_cultivos[df_cultivos['provincia'] == 'otras provincias'][['anio', 'departamento', 'provincia_id', 'cultivo']].drop_duplicates())

In [ ]:
print(df_cultivos['provincia'].value_counts().loc[['Los Andes', 'otras provincias']])

### 3.4 Exclusión de categorías no georreferenciables en `provincia`

Se detectaron dos categorías en `provincia` que no corresponden a jurisdicciones actuales: "Los Andes" (2 filas, campañas 1925 y 1937) — territorio nacional que existió hasta 1943 y luego se repartió entre Salta, Jujuy y Catamarca — y "otras provincias" (1 fila, 1924) — una categoría de agregado sin desagregar. En ambos casos `departamento` es nulo en el 100% de sus filas, por lo que no hay forma de inferir a qué provincia actual corresponden sin inventar precisión geográfica que la fuente original nunca tuvo.

Representan en conjunto 3 filas sobre 138.929 (0,002% del dataset), así que se excluyen del análisis en vez de reasignarlas arbitrariamente a una provincia moderna. El impacto en los totales nacionales es despreciable.

In [ ]:
df_cultivos = df_cultivos[~df_cultivos['provincia'].isin(['Los Andes', 'otras provincias'])]

print(df_cultivos.shape)
print(df_cultivos['provincia'].nunique())

### 3.5 Consistencia entre nombres y códigos (`_id` ↔ nombre)

Se verifica que cada `provincia_id` corresponda siempre al mismo nombre de `provincia` (y viceversa), y lo mismo para `departamento_id` y `departamento`. Un mismo código con más de un nombre asociado indicaría un error de carga, no una variante de escritura ya corregida.

In [ ]:
# provincia_id <-> provincia
print("provincia_id con más de un nombre de provincia asociado:")
print(df_cultivos.groupby('provincia_id')['provincia'].nunique().loc[lambda x: x > 1])

print("\nprovincia con más de un provincia_id asociado:")
print(df_cultivos.groupby('provincia')['provincia_id'].nunique().loc[lambda x: x > 1])

# departamento_id <-> departamento
print("\ndepartamento_id con más de un nombre de departamento asociado:")
print(df_cultivos.groupby('departamento_id')['departamento'].nunique().loc[lambda x: x > 1])

In [ ]:
# Provincia: ver los nombres detrás del código 26 y los códigos detrás de Entre Ríos / Santa Cruz
print("provincia_id 26.0 ->", df_cultivos.loc[df_cultivos['provincia_id'] == 26.0, 'provincia'].unique())
print("Entre Ríos ->", df_cultivos.loc[df_cultivos['provincia'] == 'Entre Ríos', 'provincia_id'].unique())
print("Santa Cruz ->", df_cultivos.loc[df_cultivos['provincia'] == 'Santa Cruz', 'provincia_id'].unique())

# Departamento: ver los pares de nombres en conflicto, no solo el conteo
dep_conflictivos = df_cultivos.groupby('departamento_id')['departamento'].unique()
dep_conflictivos = dep_conflictivos[dep_conflictivos.apply(len) > 1]
print(f"\n{len(dep_conflictivos)} departamento_id con más de un nombre:")
print(dep_conflictivos.head(20))

In [ ]:
mask_error = (
    ((df_cultivos['provincia'] == 'Entre Ríos') & (df_cultivos['provincia_id'] == 26.0)) |
    ((df_cultivos['provincia'] == 'Santa Cruz') & (df_cultivos['provincia_id'] == 26.0))
)
print(mask_error.sum())

In [ ]:
df_cultivos.loc[(df_cultivos['provincia'] == 'Entre Ríos') & (df_cultivos['provincia_id'] == 26.0), 'provincia_id'] = 30.0
df_cultivos.loc[(df_cultivos['provincia'] == 'Santa Cruz') & (df_cultivos['provincia_id'] == 26.0), 'provincia_id'] = 78.0

# Verificación: no debería quedar ninguna provincia con más de un id
print(df_cultivos.groupby('provincia')['provincia_id'].nunique().loc[lambda x: x > 1])

### 3.6 Normalización de nombres de departamento (renombres históricos)

El dataset abarca más de 100 años (1923-2025), período en el que varios departamentos fueron renombrados oficialmente sin cambiar de identidad geográfica (mismo `departamento_id`, distinto `departamento` según el año). Antes de decidir a qué nombre normalizar cada caso, generamos la lista completa de conflictos para investigar el nombre oficial vigente de cada uno.

In [ ]:
conflictos_df = (
    df_cultivos.groupby('departamento_id')['departamento']
    .unique()
    .loc[lambda x: x.apply(len) > 1]
    .reset_index()
    .rename(columns={'departamento': 'nombres'})
)

conflictos_df['nombres'] = conflictos_df['nombres'].apply(list)

print(f"Total de departamento_id con más de un nombre: {len(conflictos_df)}")

ruta_salida_conflictos = RUTA_INTERIM / "departamento_id_conflictos.csv"
conflictos_df.to_csv(ruta_salida_conflictos, index=False, encoding='utf-8')
print(f"Guardado en: {ruta_salida_conflictos}")

**Fuente:** Anexo II, Resolución 55/2019 - INDEC (nomenclador oficial de códigos de provincia y departamento). Documento guardado en `references/indec_resolucion_55_2019_anexo2.pdf`.

Se construye un diccionario de reemplazo (`departamento_id` → nombre oficial vigente) verificado manualmente contra la fuente, y se aplica sobre las categorías con más de un nombre detectadas en 3.5.

In [ ]:
nombres_oficiales = {
    6077.0: 'Arrecifes',
    6427.0: 'La Matanza',
    6434.0: 'Lanús',
    6588.0: '9 de Julio',
    6812.0: 'Tordillo',
    6840.0: 'Tres de Febrero',
    14007.0: 'Calamuchita',
    14042.0: 'General San Martín',
    22049.0: 'General Belgrano',
    22063.0: 'General Güemes',
    22070.0: 'Independencia',
    22084.0: 'Libertador General San Martín',
    22091.0: 'Maipú',
    22105.0: '9 de Julio',
    22140.0: 'San Fernando',
    26021.0: 'Escalante',
    38007.0: 'Cochinoca',
    38021.0: 'Dr. Manuel Belgrano',
    46028.0: 'General Felipe Varela',
    46035.0: 'Chamical',
    46056.0: 'Ángel Vicente Peñaloza',
    46070.0: 'General Juan Facundo Quiroga',
    46112.0: 'Rosario Vera Peñaloza',
    46119.0: 'San Blas de Los Sauces',
    54028.0: 'Capital',
    54042.0: 'Eldorado',
    54049.0: 'General Manuel Belgrano',
    58077.0: 'Minas',
    62028.0: 'Conesa',
    66049.0: 'General Güemes',
    70028.0: 'Capital',
    74014.0: 'Belgrano',
    74056.0: 'Juan Martín de Pueyrredón',
    82028.0: 'Constitución',
    82063.0: 'La Capital',
    86035.0: 'Banda',
    86077.0: 'General Taboada',
    86098.0: 'Juan Felipe Ibarra',
}

In [ ]:
n_cambios, _ = diagnosticar_cambio(
    df_cultivos['departamento'],
    df_cultivos['departamento_id'].map(nombres_oficiales),
)
print(f"Filas a corregir: {n_cambios}")

In [ ]:
mask = df_cultivos['departamento_id'].isin(nombres_oficiales)
df_cultivos.loc[mask, 'departamento'] = df_cultivos.loc[mask, 'departamento_id'].map(nombres_oficiales)

In [ ]:
conflictos_post = (
    df_cultivos.groupby('departamento_id')['departamento']
    .nunique()
    .loc[lambda x: x > 1]
)
print(f"departamento_id con más de un nombre restantes: {len(conflictos_post)}")

**Resultado:** 3139 filas corregidas (normalización de `departamento` a nombre oficial vigente según INDEC Resolución 55/2019, Anexo II). Verificación posterior: 0 `departamento_id` con más de un nombre — la normalización quedó completa y consistente.

### 3.7 Corrección de tipos de datos

El `.info()` inicial mostraba las columnas categóricas y de código como `object` o `float64`, sin reflejar su naturaleza real. Se ajustan:

- `cultivo`, `provincia`, `departamento` → `category` (texto de un conjunto finito de valores repetidos; reduce uso de memoria y deja explícito que son categóricas).
- `provincia_id`, `departamento_id` → `Int64` (entero *nullable* de pandas: son códigos, no mediciones continuas, pero tienen valores nulos que `int64` común no admite).

`campania`, `anio` y las columnas de medición (`superficie_sembrada_ha`, `superficie_cosechada_ha`, `produccion_tm`, `rendimiento_kgxha`) ya tenían el tipo correcto y no se modifican.

In [ ]:
columnas_a_categoria = ['cultivo', 'provincia', 'departamento']
for col in columnas_a_categoria:
    df_cultivos[col] = df_cultivos[col].astype('category')

columnas_id = ['provincia_id', 'departamento_id']
for col in columnas_id:
    df_cultivos[col] = df_cultivos[col].astype('Int64')

df_cultivos.info()

### 3.8 Diagnóstico de valores nulos remanentes

Antes de decidir el tratamiento de los valores nulos (mantenerlos, imputarlos o descartar filas), se caracteriza su patrón: si se concentran en años, provincias o cultivos particulares, o si aparecen de forma aislada — la decisión correcta depende de si el nulo responde a una causa sistemática (dato no reportado en cierta época) o es un caso puntual.

In [ ]:
# 1) provincia / provincia_id nulos (4 filas)
print("--- Filas con provincia nula ---")
print(df_cultivos[df_cultivos['provincia'].isna()][['anio', 'cultivo', 'campania']])

# 2) departamento / departamento_id nulos (¿se concentran en algún año o provincia?)
nulos_depto = df_cultivos[df_cultivos['departamento'].isna()]
print("\n--- departamento nulo: distribución por año (top 10) ---")
print(nulos_depto['anio'].value_counts().sort_index().tail(10))

# 3) columnas de medición (el detalle completo se ve en la celda siguiente)
cols_medicion = ['superficie_cosechada_ha', 'produccion_tm', 'rendimiento_kgxha']

In [ ]:
for provincia, count in nulos_depto['provincia'].value_counts().items():
    print(f"{provincia}: {count}")

In [ ]:
patron_nulos = df_cultivos[cols_medicion].isna().value_counts()
for combinacion, count in patron_nulos.items():
    print(f"{dict(zip(cols_medicion, combinacion))}: {count}")

In [ ]:
completos = df_cultivos[df_cultivos[cols_medicion].notna().all(axis=1)].copy()
completos['rendimiento_calculado'] = completos['produccion_tm'] * 1000 / completos['superficie_cosechada_ha']
diferencia = (completos['rendimiento_calculado'] - completos['rendimiento_kgxha']).abs()
print(diferencia.describe())

In [ ]:
casos_raros = df_cultivos[
    df_cultivos['rendimiento_kgxha'].notna()
    & (
        (df_cultivos['superficie_cosechada_ha'].isna() & df_cultivos['produccion_tm'].isna())
        | (df_cultivos['superficie_cosechada_ha'].notna() & df_cultivos['produccion_tm'].isna())
    )
]
print(f"Casos anómalos encontrados: {len(casos_raros)}")

nulos_rendimiento_recuperable = df_cultivos[
    df_cultivos['superficie_cosechada_ha'].notna()
    & df_cultivos['produccion_tm'].notna()
    & df_cultivos['rendimiento_kgxha'].isna()
]

RUTA_INTERIM.mkdir(parents=True, exist_ok=True)
nulos_rendimiento_recuperable.to_csv(RUTA_INTERIM / 'grupo_31_nulos_rendimiento.csv', index=False, encoding='utf-8')
casos_raros.to_csv(RUTA_INTERIM / 'casos_anomalos_rendimiento.csv', index=False, encoding='utf-8')
print("Archivos guardados en data/interim")

Recuperación de valores calculables y documentación de nulos remanentes

Se verificó que `rendimiento_kgxha = producción_tm × 1000 / superficie_cosechada_ha` es la fórmula real de la fuente (diferencia mediana ≈ 0 contra ~124.000 filas completas). Con la fórmula confirmada, se recuperan por cálculo directo los casos donde exactamente una de las tres columnas falta y las otras dos están presentes (209 filas en total):

- 177 filas: se calcula `superficie_cosechada_ha`
- 31 filas: se calcula `rendimiento_kgxha`
- 1 fila: se calcula `produccion_tm`

Los 6.213 casos restantes (6.187 con las tres columnas vacías + 24 + 2 con dos columnas vacías) quedan como `NaN`: no hay forma de derivarlos matemáticamente (faltan dos de tres valores), y se documentan como una laguna real de la fuente histórica, concentrada mayormente en campañas anteriores a 1968.

In [ ]:
# Máscaras calculadas sobre el estado original, antes de modificar nada
falta_cosechada = df_cultivos['superficie_cosechada_ha'].isna() & df_cultivos['produccion_tm'].notna() & df_cultivos['rendimiento_kgxha'].notna()
falta_rendimiento = df_cultivos['superficie_cosechada_ha'].notna() & df_cultivos['produccion_tm'].notna() & df_cultivos['rendimiento_kgxha'].isna()
falta_produccion = df_cultivos['superficie_cosechada_ha'].notna() & df_cultivos['produccion_tm'].isna() & df_cultivos['rendimiento_kgxha'].notna()

print(f"Filas a calcular -> cosechada: {falta_cosechada.sum()}, rendimiento: {falta_rendimiento.sum()}, producción: {falta_produccion.sum()}")

df_cultivos.loc[falta_cosechada, 'superficie_cosechada_ha'] = (
    df_cultivos.loc[falta_cosechada, 'produccion_tm'] * 1000 / df_cultivos.loc[falta_cosechada, 'rendimiento_kgxha']
)
df_cultivos.loc[falta_rendimiento, 'rendimiento_kgxha'] = (
    df_cultivos.loc[falta_rendimiento, 'produccion_tm'] * 1000 / df_cultivos.loc[falta_rendimiento, 'superficie_cosechada_ha']
)
df_cultivos.loc[falta_produccion, 'produccion_tm'] = (
    df_cultivos.loc[falta_produccion, 'rendimiento_kgxha'] * df_cultivos.loc[falta_produccion, 'superficie_cosechada_ha'] / 1000
)

# Verificación: los nulos deberían bajar exactamente en 209 filas en total
print(df_cultivos[cols_medicion].isnull().sum())

**Nota:** el cálculo de `superficie_cosechada_ha` a partir de `produccion_tm` y `rendimiento_kgxha` (más arriba) generó 30 filas donde el resultado supera levemente a `superficie_sembrada_ha` — siempre por márgenes menores al 0,03% (ej. 47.000 vs. 47.007,35 ha), y siempre en maíz, campaña 1923. No es una inconsistencia real de la fuente: es un artefacto de redondeo, porque `rendimiento_kgxha` se reportó como entero, y al despejar `cosechada` con ese valor ya redondeado el resultado hereda ese margen de error. Dado que `superficie_cosechada_ha` no puede superar físicamente a `superficie_sembrada_ha` (regla establecida en 3.1), se acota el valor calculado a ese techo en estos 30 casos puntuales.

In [ ]:
exceso_redondeo = df_cultivos['superficie_cosechada_ha'] > df_cultivos['superficie_sembrada_ha']
print(f"Filas a acotar: {exceso_redondeo.sum()}")

df_cultivos.loc[exceso_redondeo, 'superficie_cosechada_ha'] = df_cultivos.loc[exceso_redondeo, 'superficie_sembrada_ha']

# Verificación: no debería quedar ninguna fila con cosechada > sembrada
print((df_cultivos['superficie_cosechada_ha'] > df_cultivos['superficie_sembrada_ha']).sum())

### 3.9 Categorías de agregado sin desagregar en `departamento`

Se investigaron tres valores de "cajón de sastre" en `departamento`: "otros departamentos" (61 filas) y "otros partidos" (4 filas) no tienen `departamento_id` real asociado. "sin definir" (220 filas) sí tenía un `departamento_id`, pero los 20 códigos involucrados terminan en 000 (ej. 06000, 14000) — se verificó contra el nomenclador oficial INDEC (Resolución 55/2019, Anexo II) que ningún departamento real usa ese sufijo; es un código de "total provincial sin desagregar", no un departamento verdadero.

Los tres casos representan la misma situación ya documentada en 3.8 (dato de producción real a nivel provincial, sin desglose por departamento), así que se unifican convirtiendo `departamento` y `departamento_id` a `NaN` en los tres, en vez de mantener tres formas distintas de representar el mismo vacío.

In [ ]:
print("Filas con departamento == 'otros departamentos':", (df_cultivos['departamento'] == 'otros departamentos').sum())
print(df_cultivos[df_cultivos['departamento'] == 'otros departamentos'][['anio', 'provincia', 'cultivo']].to_string())

# Lista completa de departamentos únicos, para revisar a simple vista si hay
# otras categorías de agregado que no hayamos detectado todavía
RUTA_INTERIM.mkdir(parents=True, exist_ok=True)
pd.Series(sorted(df_cultivos['departamento'].dropna().unique())).to_csv(
    RUTA_INTERIM / 'departamentos_unicos.csv', index=False, header=['departamento']
)
print("Lista completa guardada en data/interim/departamentos_unicos.csv")

In [ ]:
valores_agregado = ['otros departamentos', 'otros partidos', 'sin definir']

for valor in valores_agregado:
    filas = df_cultivos[df_cultivos['departamento'] == valor]
    print(f"'{valor}': {len(filas)} filas | departamento_id nulo en {filas['departamento_id'].isna().sum()} de {len(filas)}")

print("\nDistribución por provincia:")
print(df_cultivos[df_cultivos['departamento'].isin(valores_agregado)]['provincia'].value_counts())

In [ ]:
sin_definir = df_cultivos[df_cultivos['departamento'] == 'sin definir']
ids_sin_definir = sin_definir['departamento_id'].unique()
print(f"departamento_id únicos en 'sin definir': {len(ids_sin_definir)}")

# ¿Esos mismos id aparecen en otras filas con un nombre real ya completado?
recuperables = df_cultivos[
    df_cultivos['departamento_id'].isin(ids_sin_definir) & (df_cultivos['departamento'] != 'sin definir')
]
print(f"\nFilas con esos id y nombre real: {len(recuperables)}")
print(recuperables.groupby('departamento_id')['departamento'].unique())

In [ ]:
print(sorted(ids_sin_definir))

In [ ]:
valores_agregado = ['otros departamentos', 'otros partidos', 'sin definir']
mask_agregado = df_cultivos['departamento'].isin(valores_agregado)

df_cultivos.loc[mask_agregado, 'departamento'] = pd.NA
df_cultivos.loc[mask_agregado, 'departamento_id'] = pd.NA

# Verificación
print("Quedan valores de agregado en departamento:", df_cultivos['departamento'].isin(valores_agregado).sum())
print("Nulos en departamento:", df_cultivos['departamento'].isna().sum())
print("Nulos en departamento_id:", df_cultivos['departamento_id'].isna().sum())

## 4. Cierre de la fase de Limpieza

Después de auditar y corregir más de un siglo de registros (1923-2025) para 8 cultivos, el dataset queda en condiciones de sostener el análisis que motiva todo el proyecto: entender cómo el clima histórico impactó el rendimiento agrícola en Argentina.

En el camino corregimos errores de codificación (mojibake), dos columnas de superficie invertidas, nombres de departamento inconsistentes o históricamente renombrados, y recuperamos por cálculo 209 valores que sí eran deducibles a partir de las otras columnas. Quedan documentadas, y no inventadas, unas 6.200 filas sin rendimiento registrado — concentradas en campañas anteriores a 1968, cuando el relevamiento oficial era menos sistemático. Esa laguna real de la fuente histórica hay que tenerla presente en cualquier comparación que incluya esas décadas, pero no compromete el análisis desde los años 90 en adelante, que es donde además vamos a poder cruzar estos datos con información climática.

El dataset limpio y verificado se guarda en `data/processed`, marcando el cierre de esta etapa. A partir de este punto, cualquier notebook posterior (fusión con clima, exploración, modelado) debería partir de este archivo en vez de repetir el pipeline de limpieza completo.

In [ ]:
RUTA_PROCESSED = RUTA_PROYECTO / "data" / "processed"
RUTA_PROCESSED.mkdir(parents=True, exist_ok=True)

ruta_salida = RUTA_PROCESSED / "cultivos_procesado.csv"
df_cultivos.to_csv(ruta_salida, index=False, encoding='utf-8')

print(f"Guardado en: {ruta_salida}")
print(f"Dimensiones finales: {df_cultivos.shape}")